# Sycophancy tone experiment on Colab (GPU)

**Before running:** In Colab choose **Runtime → Change runtime type → GPU** (e.g. T4) and save.

This notebook: (1) Installs Ollama and runs it on the Colab GPU, (2) Clones sycophancy-eval and loads this repo, (3) Runs the full experiment via `run_experiment.py` (generate → inference → aggregate → plot). Download results from the Files panel or copy to Drive.

## 1. Install Ollama and start server (uses GPU)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# Start Ollama in background (uses GPU when available)
proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
print("Ollama server started.")

## 2. Pull a model (choose one; smaller = faster on T4)

Examples: `llama3.2` (~2B, fast), `llama3.2:3b`, `mistral`, `phi3`.

In [ ]:
!ollama pull llama3.2

## 3. Clone repos and install Python deps

In [ ]:
import os
from pathlib import Path

ROOT = Path("/content/sycophancy-tone-extension")

# Clone meg-tong datasets (for answer.jsonl)
if not Path("/content/sycophancy-eval").exists():
    !git clone --depth 1 https://github.com/meg-tong/sycophancy-eval.git /content/sycophancy-eval

# This repo: either unzip an uploaded zip, or clone from GitHub
if not ROOT.exists():
    # If you uploaded sycophancy-tone-extension.zip to Colab Files, run:
    # !unzip -q /content/sycophancy-tone-extension.zip -d /content
    !unzip -q -o /content/sycophancy-tone-extension.zip -d /content 2>/dev/null || true
if not ROOT.exists():
    raise SystemExit("Upload this repo as a zip to Colab (Files panel), name it sycophancy-tone-extension.zip, then re-run this cell.")

os.chdir(ROOT)
!pip install -q tqdm langchain-openai langchain-core
print("Repo and deps ready.")

## 3b. Verify Ollama is reachable (avoids "Connection error" in results)

Run this **before** the experiment. If it fails, re-run the "Install Ollama and start server" and "Pull a model" cells, then run this again.

In [ ]:
import urllib.request
import json

try:
    req = urllib.request.Request("http://127.0.0.1:11434/api/tags", method="GET")
    with urllib.request.urlopen(req, timeout=5) as r:
        data = json.loads(r.read().decode())
    models = [m["name"] for m in data.get("models", [])]
    print("Ollama is reachable. Models:", models if models else "(none pulled yet)")
    if not models:
        print("Run the 'Pull a model' cell above, then re-run this cell.")
except Exception as e:
    raise SystemExit(f"Ollama not reachable: {e}. Re-run the 'Install Ollama and start server' cell and wait a few seconds.")

## 4. Run full experiment (generate → inference → aggregate → plot)

Uses `run_experiment.py` so the pipeline matches local runs. Add `--limit 100` for a quick test.

In [ ]:
!python run_experiment.py \
  --datasets-dir /content/sycophancy-eval/datasets \
  --models ollama/llama3.2 \
  --base-url http://localhost:11434/v1 \
  --regenerate

## 5. Results

Outputs are in `results/sycophancy_by_tone.html` and `results/summary_by_tone.json`. Download them from the Files panel (left sidebar), or use the optional Drive step below.

In [ ]:
# run_experiment above already wrote results; list them:
!ls -la results/

## 6. (Optional) Copy results to Google Drive

In [ ]:
# Uncomment to mount Drive and copy results:
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r results /content/drive/MyDrive/sycophancy_tone_results
# print("Copied results to Drive.")